In [ ]:
import json
import os
import pandas as pd
%load_ext autoreload
%autoreload 2
from dotenv import load_dotenv
from constant import *
from GeminiModel import GeminiModel
load_dotenv()
from LlmOutputLabelConverter import LlmOutputLabelConverter
from Model import Model
from ChatGpt4Model import ChatGpt4Model

In [ ]:

def parse_batch_output(model:Model, file, model_suffix):
    df = pd.concat([detect_train_df, detect_test_df], ignore_index=True) if model.task_type == 'detect' else pd.concat([classify_train_df, classify_test_df], ignore_index=True)
    batch_dataset = Dataset.from_pandas(df)
    print(f'parsing file {file}')
    with open(file, 'r', encoding='utf-8') as json_output_file:
        ids = []
        ids_errors = []
        predicted_labels = []
        raw_predicted_labels = []
        for line in json_output_file:
            output = json.loads((line.strip()))
            try:
                if isinstance(model, GeminiModel):
                    sample_id = int(output['key'])
                    if 'response' not in output or output['response'] is None:
                        ids_errors.append(sample_id)
                        continue
                    raw_predicted_label = output['response']['candidates'][-1]['content']['parts'][-1]['text']
                elif isinstance(model, ChatGpt4Model):
                    sample_id = int(output['custom_id'])
                    raw_predicted_label = output['response']['body']['choices'][-1]['message']['content']
                else:
                    raise RuntimeError(f'Unknown model type: {model.model_uri}')
            except KeyError as e:
                raise RuntimeError(f'{line}: {e}')
            ids.append(sample_id)
            raw_predicted_labels.append(raw_predicted_label)
            predicted_labels.append(model.output_label_converter.convert_label(raw_predicted_label))
        model.append_into_merged_file(model_suffix, batch_dataset, ids, predicted_labels, raw_predicted_labels)
        print("Rows parsed {}/{}".format(len(ids), len(ids_errors) + len(ids)))
        if len(ids_errors) > 0:
            print(f'Errors ids: {ids_errors}')


def get_completed_status_set(model_uri):
    if 'gemini' in model_uri:
        return {'JOB_STATE_SUCCEEDED', 'JOB_STATE_FAILED', 'JOB_STATE_CANCELLED', 'JOB_STATE_EXPIRED'}
    elif 'gpt' in model_uri:
        return {'failed', 'completed', 'expired', 'cancelling','cancelled' }
    else:
        return {}

def retrieve_gemini_status_and_result_if_available(model:GeminiModel, job_id):
    batch_job = model.model.batches.get(name=job_id)
    if batch_job.state.name == 'JOB_STATE_SUCCEEDED':
        if batch_job.dest and batch_job.dest.file_name:
            result_file_name = batch_job.dest.file_name
            job_file_response = model.model.files.download(file=result_file_name)
            return [batch_job.state.name, job_file_response.decode('utf-8') ]
    return [batch_job.state.name, None]


def retrieve_gpt_status_and_result_if_available(model: ChatGpt4Model, job_id):
    job_status_response = model.client.batches.retrieve(job_id)
    job_status = job_status_response.status
    if job_status == 'completed':
        file_response = model.client.files.content(job_status_response.output_file_id)
        return [job_status, file_response.text]
    return [job_status, None]



In [ ]:

JOB_FILE = f'{os.getenv("CACHE_DIRECTORY")}/output/batch/job.csv'
classification_label_set = set(classify_train_dataset['label']) | set(classify_test_dataset['label'])
CLASSIFICATION_LABEL_CONVERTER = LlmOutputLabelConverter(classification_label_set, DEFAULT_CLASSIFICATION_CLASS)
DETECTION_LABEL_CONVERTER = LlmOutputLabelConverter({'yes', 'no'}, DEFAULT_DETECTION_CLASS)

# parse_batch_output(GeminiModel('detect', 'models/gemini-2.0-flash', output_label_converter, True), '/home/cs/grad/islams32/dev/project/academic/technical-debt/cache/output/batch/output/detect_gemini-2.0-flash-2-shot.jsonl', '2-shot')
# parse_batch_output(ChatGpt4Model('detect', 'gpt-5', CLASSIFICATION_LABEL_CONVERTER, True, True), '/Users/shahidul/dev/project/technical-debt/cache/output/batch/output/detect_gpt-5-0-shot.jsonl', '0-shot')

In [ ]:
import time

tryAgain = True
while tryAgain:
    tryAgain = False
    job_df = pd.read_csv(JOB_FILE, dtype={"status": "string"})
    for idx, row in job_df.iterrows():
        model_uri = row["model_uri"]
        completed_states = get_completed_status_set(model_uri)
        if row["status"] not in completed_states:
            tryAgain = True
            task_type = row['task_type']
            if task_type == 'detect':
                output_label_converter = DETECTION_LABEL_CONVERTER
            elif task_type == 'classify':
                output_label_converter = CLASSIFICATION_LABEL_CONVERTER
            else:
                raise Exception(f'Unknown task type: {task_type}')
            if  'gemini' in  model_uri:
                model = GeminiModel(task_type, model_uri, output_label_converter, True, True)
                job_response = retrieve_gemini_status_and_result_if_available(model, row['job_id'])

            elif 'gpt' in model_uri:
                model = ChatGpt4Model(task_type, model_uri, output_label_converter, True, True)
                job_response = retrieve_gpt_status_and_result_if_available(model, row['job_id'])
            else:
                raise Exception(f'Unknown model uri: {model_uri}')

            job_name = row['job_name']
            updated_job_status, result_json = job_response
            if result_json:
                output_file_name = row['input_file'].replace('/input/', '/output/')
                os.makedirs(os.path.dirname(output_file_name), exist_ok=True)
                with open(output_file_name, 'w') as output_file:
                    output_file.write(result_json)
                model_suffix = job_name[job_name.rfind('-',0, len(job_name) - len('-shot')) + 1:]
                parse_batch_output(model, output_file_name, model_suffix)
            print(f"Job status detail {updated_job_status}")
            job_df.at[idx, "status"] = updated_job_status
            job_df.at[idx, "updated_at"] = pd.Timestamp.now()
            job_df.to_csv(JOB_FILE, index=False)

    if tryAgain:
        time.sleep(30)
